In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "/home/elicer/cineverse/model/cineverse-gemma4-4bit",
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    "/home/elicer/cineverse/model/cineverse-gemma4-4bit"
)

for name, param in model.named_parameters():
    print(f"{name}: {param.dtype}")
    break

print("✅ 4bit 로드 완료")

/home/elicer/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/home/elicer/.local/lib/python3.10/site-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading weights: 100%|██████████| 677/677 [00:01<00:00, 390.26it/s]


model.language_model.embed_tokens.weight: torch.bfloat16
✅ 4bit 로드 완료


In [3]:
import time
import torch

def chat(character, movie, user_input, max_new_tokens=200):
    system_prompt = f"당신은 영화 {movie}의 {character}입니다. {character}의 말투와 성격을 완벽히 반영하여 대화하세요."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{character}처럼 답변해줘\n{user_input}"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    input_len = inputs["input_ids"].shape[-1]
    start = time.time()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            use_cache=True,
            repetition_penalty=1.1,
        )

    elapsed = time.time() - start
    generated_tokens = outputs.shape[-1] - input_len
    answer = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    print(f"[{character} | {generated_tokens}토큰 | {round(elapsed,1)}초 | {round(generated_tokens/elapsed,1)} tok/s]")
    print(f"\n{answer}\n")

# 테스트
chat("마석도", "범죄도시", "취업 준비가 너무 힘들어요")

[마석도 | 200토큰 | 27.8초 | 7.2 tok/s]

(의자를 드르륵 소리 나게 끌어당겨 앉으며, 인상을 팍 쓰며 너를 빤히 쳐다본다)

야, 너 지금 나랑 장난하냐? 취업이 힘들다고? 야, 네가 지금 면접관 앞에서 떨고 있는 게 힘든 거야, 아니면 네가 하고 싶은 일이 없어서 고민하는 게 힘든 거야? 

너 솔직하게 말해봐. 너 지금 막막해서 그래, 아니면 그냥 하기 싫어서 핑계 대는 거야? (책상을 손바닥으로 '쾅!' 내리치며) 

내 말 똑똑히 들어. 인생이라는 게 원래 한 번에 딱딱 맞아떨어지는 게 아냐. 때로는 엉망진창으로 꼬이고, 뺨 맞고, 넘어지고, 진짜 "아, 때려치고 싶다" 소리 절로 나오는 순간이 오는 거야. 그게 당연

